# U10 练习题

完成以下练习，过关标准：练习 10.3 训练 loss < 1.0 并能复现训练样本。

---

In [ ]:
import torch
import torch.nn as nn
import random
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']

EMBED_DIM, HIDDEN_DIM = 64, 128
BATCH_SIZE = 8
torch.manual_seed(0); random.seed(0)

raw_pairs = [
    ('我爱你', 'i love you'),
    ('我喜欢猫', 'i like cats'),
    ('他在看书', 'he is reading a book'),
    ('今天天气真好', 'the weather is nice today'),
    ('谢谢你', 'thank you'),
    ('对不起', 'i am sorry'),
    ('我饿了', 'i am hungry'),
    ('明天见', 'see you tomorrow'),
]

## 练习 10.1：手写带 packing 的 Encoder

补全 Encoder：
- Embedding 必须设 `padding_idx=PAD`
- 用 `pack_padded_sequence` + GRU
- 只返回 hidden（context vector）

**自检要点**：随便给两条不等长 src，packing 后 hidden 应该和『按真实长度只算前 n 步』的纯手工版本数值一致（不会被 pad 污染）。

In [ ]:
def tokenize_zh(t): return list(t)
def tokenize_en(t): return t.lower().split()

class Vocab:
    def __init__(self, token_lists):
        c = Counter()
        for ts in token_lists: c.update(ts)
        self.itos = list(SPECIALS) + [t for t, _ in c.most_common()]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, ts): return [self.stoi.get(t, UNK) for t in ts]
    def decode(self, ids): return [self.itos[i] for i in ids]

src_vocab = Vocab([tokenize_zh(zh) for zh, en in raw_pairs])
tgt_vocab = Vocab([tokenize_en(en) for zh, en in raw_pairs])


class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO 1: Embedding（注意 padding_idx）
        # TODO 2: GRU（batch_first=True）
        pass

    def forward(self, src, src_len):
        # TODO 3: embed -> pack -> gru -> 返回 hidden
        pass


# 自测
enc = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
src = torch.tensor([[5, 6, 7, 2, 0, 0],
                    [8, 9, 2, 0, 0, 0]])
src_len = torch.tensor([4, 3])
h = enc(src, src_len)
assert h.shape == (1, 2, HIDDEN_DIM), f'shape 错: {h.shape}'
print('shape OK:', h.shape)

## 练习 10.2：手写 greedy decode 翻译函数

假设已经有训练好的 `model`（含 `encoder` 和 `decoder` 两个子模块），实现 `translate(model, sentence, max_len=20)`：
1. 把 sentence 转成 src tensor 和 src_len
2. 过 Encoder 拿 hidden
3. 自回归循环：每步取 argmax，遇 EOS 或达 max_len 停
4. 返回空格分隔的英文译文字符串

**易错点**：
- 输入 Decoder 的 token shape 必须是 `(1, 1)`，不是 `(1,)` 或 `()`
- 不要把 `<sos>` 或 `<eos>` 加到结果里

In [ ]:
def translate(model, sentence, max_len=20):
    model.eval()
    with torch.no_grad():
        # TODO 1: 句子 -> src_ids（加 EOS 不加 SOS）-> tensor
        # TODO 2: encoder 得 hidden
        # TODO 3: input_tok = SOS（shape (1, 1)）
        # TODO 4: 循环 max_len 次，argmax 取下一个 token，遇 EOS 跳出
        # TODO 5: decode result_ids 成字符串
        pass

# 写完后跑下面的练习 10.3 训练完，再回来调用 translate

## 练习 10.3：跑通完整训练 + 推理（过关任务）

把上面两题的 Encoder / translate 结合，完成完整训练循环，让模型在小语料上**过拟合**到 loss < 1.0，并能复现训练样本。

**过关标准**：
- 最终 epoch 平均 loss < 1.0
- `translate(model, '我爱你')` 输出 `i love you`（或非常接近）
- 至少 5/8 训练样本能正确复现

**调试提示**：
- loss 不下降：检查 `pack_padded_sequence` 是否传了 `src_len.cpu()`、collate_fn 是否按 src 长度降序排
- loss 直接 nan：加梯度裁剪 `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`
- 推理乱码：检查 `Decoder` 输入 shape 是否 `(B, 1)`、是否真的把 hidden 传进 GRU
- loss 卡在 2-3 不下降：检查 loss 是否用了 `ignore_index=PAD`，错位切片是否对

In [ ]:
# === Decoder 已给出 ===
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(output.squeeze(1))
        return logits, hidden


# === TODO: 补全 Seq2Seq ===
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        # TODO: 参考 lesson 第 5 节
        pass


# === TODO: 补全数据管道 ===
def sentence_to_ids(sent, vocab, tok, add_sos=False, add_eos=True):
    # TODO
    pass

def pad_sequence(ids_list, pad_id=PAD):
    # TODO: 补 PAD 到 batch 内最长
    pass

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        # TODO: 返回 (src_ids, tgt_ids)
        pass

def collate_fn(batch):
    # TODO: 按 src 长度降序、记录 src_len、padding，返回 (src, src_len, tgt)
    pass


# === TODO: 训练循环 ===
encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
loader = DataLoader(TranslationDataset(raw_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
V_tgt = len(tgt_vocab)

for epoch in range(1, 101):
    model.train()
    total_loss, n = 0.0, 0
    for src, src_len, tgt in loader:
        # TODO: zero_grad, forward, loss, backward, clip_grad, step, 累加 total_loss
        pass
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | loss={total_loss/max(n,1):.4f}')

# 推理验证
for zh, en in raw_pairs:
    print(f'{zh}  ->  {translate(model, zh)}   (gold: {en})')

## 练习 10.4：默写练习（不看资料）

**关闭 lesson.ipynb**，凭记忆完成。

In [ ]:
# Q1: pack_padded_sequence 有哪两个硬性要求？
# A1:

# Q2: 处理 padding 有三个地方需要做『屏蔽』，分别是哪三处？分别防止什么问题？
# A2:
#   1) ...
#   2) ...
#   3) ...

# Q3: greedy decode 的两个终止条件是什么？
# A3:

# Q4: 训练时 src 不需要 SOS、tgt 两端都要加，为什么？
# A4:

# Q5: Baseline Seq2Seq 的『信息瓶颈』具体指什么？为什么句子越长效果越差？
# A5:

# Q6: U11 的 Attention 怎么改造 Encoder 输出和 Decoder 输入来突破这个瓶颈？
# A6:

# Q7: 为什么训练 loss 已经很低，但翻译新句子（训练集没有的）效果还是很差？
# A7:

print('默写完成后，对照 lesson.ipynb 检查')